# Pneumonia Classification — thực nghiệm thật
Chạy từ bản sao project trên máy hoặc Colab GPU. Notebook dùng cùng mã nguồn CLI; không chứa số liệu dựng sẵn. Trên Colab, đặt toàn bộ project trong `/content/tgmt` trước khi chạy. Chọn Runtime → GPU.

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/content/tgmt') if Path('/content/tgmt').exists() else Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
assert Path('src/train.py').exists(), 'Cần đặt notebook trong project'
os.environ['TORCH_HOME'] = str(ROOT / '.torch')
def run(*args): subprocess.run([sys.executable, *args], check=True)


In [ ]:
run('-m', 'pip', 'install', '-r', 'requirements.txt')
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available())

## Dataset và thống kê
Nguồn: Kermany et al. (2018), doi:10.17632/rscbjbr9sj.2. Chỉ dùng ảnh X-quang thật. Nếu download yêu cầu đăng nhập, tải ZIP vào `data/chest-xray-pneumonia.zip`.

In [ ]:
run('scripts/download_data.py')
run('-m', 'src.dataset')
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('results/dataset_counts.csv'))
display(Image(filename='results/dataset_distribution.png'))

## Train classifier rồi fine-tune
Cấu hình YAML điều khiển LR, batch, epochs, patience và weighting. Không điều chỉnh cấu hình dựa vào test. Nếu chạy lại, dùng output mới để giữ provenance.

In [ ]:
run('-m', 'src.train', '--config', 'configs/resnet50.yaml')

In [ ]:
run('-m', 'src.train', '--config', 'configs/densenet121.yaml')

In [ ]:
for name in ['resnet50', 'densenet121']:
    run('-m', 'src.evaluate', '--checkpoint', f'checkpoints/best_{name}.pth')
run('-m', 'src.report')
display(pd.read_csv('results/model_comparison.csv'))

In [ ]:
for name in ['resnet50', 'densenet121']:
    display(pd.read_csv(f'logs/history_{name}.csv'))
    for suffix in ['loss_curve','accuracy_curve','precision_recall','f1_curve','auc_curve']:
        display(Image(filename=f'results/plots/{name}_{suffix}.png'))
display(Image(filename='results/roc/model_comparison_roc.png'))

## Phân tích TP/TN/FP/FN
Kiểm tra vùng phổi, chữ/marker, viền và background. Không suy diễn heatmap thành bằng chứng tổn thương. Ghi nhận nhóm không có mẫu, không tạo case giả.

In [ ]:
for name in ['resnet50','densenet121']:
    for case in ['TP','TN','FP','FN']:
        images = sorted(Path(f'results/gradcam/{name}').glob(f'{case}_*_overlay.png'))
        print(name, case, 'Không có mẫu' if not images else '')
        for image in images: display(Image(filename=str(image)))

## Báo cáo / demo
Báo cáo: `results/report.md`; slide: `results/presentation.pptx`. Chạy demo ở terminal: `python -m streamlit run app/app.py`. Kết luận cần cân nhắc AUC, sensitivity, specificity, F1, FN và giới hạn một nguồn dữ liệu/một seed. Grad-CAM không thay thế chẩn đoán lâm sàng.